In [1]:
#Pvt PO
import pandas as pd

In [2]:
# PO_df = pd.read_excel("PTC draft 2025-12-19.xlsx")

In [3]:
import pandas as pd

PO_df = pd.read_excel("PTC draft 2025-12-19.xlsx")

# ======================================================
# 1. Build base PN–Order table
# ======================================================
df = (
    PO_df[['PN AL78', 'Order No']]
    .dropna()
    .drop_duplicates()
    .sort_values(['PN AL78', 'Order No'])
    .reset_index(drop=True)
)

# ======================================================
# 2. Line show (count per PN)
# ======================================================
df['Line show'] = df.groupby('PN AL78')['Order No'].transform('count')

# ======================================================
# 3. chk = "chg" only on first row of each PN
# ======================================================
df['chk'] = ''
df.loc[df.groupby('PN AL78').head(1).index, 'chk'] = 'chg'

# ======================================================
# 4. PO = comma-separated Order No (first row only)
# ======================================================
po_map = (
    df.groupby('PN AL78')['Order No']
    .apply(lambda x: ','.join(x.astype(str)))
)

df['PO'] = ''
first_idx = df.groupby('PN AL78').head(1).index
df.loc[first_idx, 'PO'] = df.loc[first_idx, 'PN AL78'].map(po_map)

# ======================================================
# 5. To row (Excel row reference of LAST occurrence)
# ======================================================

# Get last index per PN AL78
last_row_map = (
    df.groupby('PN AL78')
      .apply(lambda x: x.index.max())
)

# Build Excel row reference (B + row number)
to_row_map = last_row_map.astype(int).add(2).astype(str).radd('B')

df['To row'] = ''
df.loc[first_idx, 'To row'] = df.loc[first_idx, 'PN AL78'].map(to_row_map)


# ======================================================
# 6. Insert 2 empty columns after Order No
# ======================================================
df.insert(2, '', '')
df.insert(3, ' ', '')

# ======================================================
# 7. RIGHT SIDE SUMMARY TABLE
# ======================================================
summary_df = (
    df[['PN AL78', 'Line show']]
    .drop_duplicates()
    .rename(columns={'Line show': 'Lines'})
    .assign(Lines=lambda x: x['Lines'].astype('Int64'))
    .reset_index(drop=True)
)

C:\Users\Brandon\AppData\Local\Temp\ipykernel_20608\812933588.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.index.max())


In [4]:
# Create spacer columns between tables
spacer = pd.DataFrame({'': [''] * len(df)})

# Pad summary_df to same length as df
summary_padded = summary_df.reindex(range(len(df)))

# Combine everything
final_df = pd.concat([df, spacer, summary_padded], axis=1)

print(final_df)


      PN AL78  Order No        Line show  chk  \
0      107460    759472                4  chg   
1      107460    759574                4        
2      107460    763202                4        
3      107460    763212                4        
4      107695    759574                5  chg   
...       ...       ... .. ..        ...  ...   
2140  S  2268    759574                6        
2141  S  2268    762379                6        
2142  S  2268    763145                6        
2143  S  2268    763202                6        
2144  S  2268    763212                6        

                                      PO To row    PN AL78  Lines  
0            759472,759574,763202,763212     B5     107460      4  
1                                                   107695      5  
2                                                   108722      3  
3                                                   109319      1  
4     759574,762754,763145,763202,763212    B10     109660      1  
...

In [5]:
final_df.to_excel("Pvt PO draft.xlsx", index=False)
